# 01 — Ablasi fitur: 19 penuh vs 13 geometri

**Pertanyaan yang dijawab notebook ini.** Model Neurogaze saat ini memakai 19 fitur yang
diekstrak dari citra scanpath Carette. Enam di antaranya membaca kanal warna citra:
`vel_mean`, `vel_std` dari kanal R; `acc_mean`, `jerk_mean` dari G dan B; serta
`fixation_ratio` dan `saccade_ratio` dari perbandingan antar kanal. Carette meng-encode
kinematika pandangan ke dalam warna, dan enam fitur itu membacanya kembali.

**Kenapa ini menghalangi produk.** Kamera tablet menghasilkan deret `(x, y, t)`, bukan PNG
ber-encoding Carette. Untuk memakai keenam fitur itu di produk, kita harus mereplikasi
skema normalisasi Carette persis — dan skema itu tidak terdokumentasi. Tebakan yang meleset
menempatkan fitur di luar skala latih **tanpa memunculkan error apa pun**: model tetap
mengeluarkan angka, hanya saja angkanya tidak bermakna. Kegagalan diam-diam seperti itu
tidak bisa diterima pada alat yang mengeluarkan rekomendasi rujukan.

Tiga belas fitur sisanya murni geometri sebaran tinta — hanya membaca posisi piksel,
tidak pernah warnanya. Fitur ini dapat dihitung dari scanpath perangkat apa pun.

**Yang diuji:** apakah membuang enam fitur warna merugikan performa secara berarti.

## Aturan keputusan (ditetapkan sebelum melihat hasil)

Ditulis lebih dahulu supaya keputusan tidak menyesuaikan diri dengan angka yang keluar.

| # | Kriteria | Ambang |
|---|---|---|
| **K1** | AUC level anak, regresi logistik, 13 fitur geometri | **≥ 0,80** |
| **K2** | Batas bawah CI 95% selisih AUC (geometri − penuh) | **> −0,05** |

**K1** menjaga performa absolut tetap layak untuk triase. **K2** adalah uji
non-inferioritas: selisih boleh negatif, asalkan bukti tidak mendukung penurunan
lebih dari 0,05 AUC.

Selisih diuji dengan **bootstrap berpasangan** — kedua set fitur dievaluasi pada anak
yang sama dan lipatan yang sama, jadi satu penarikan indeks dipakai untuk menghitung
kedua AUC lalu selisihnya yang dikumpulkan. Membandingkan dua CI yang tumpang tindih
bukan uji yang benar untuk selisih.

- **Lulus keduanya** → kunci model geometri sebagai model produk. Pipeline webcam bisa
  dibangun langsung di atasnya.
- **Gagal salah satu** → verifikasi dulu skema encoding Carette sebelum memutuskan.

Semua evaluasi memakai **GroupKFold dengan ID partisipan sebagai grup**, dan metrik utama
dihitung pada **unit anak**, bukan unit citra — sesuai unit keputusan produk.

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

sys.path.insert(0, str(Path.cwd() if Path.cwd().name == "research" else Path.cwd() / "research"))

import features as F
import evaluate as E

np.random.seed(E.RANDOM_STATE)

HASIL = F.ROOT / "research" / "hasil"
HASIL.mkdir(parents=True, exist_ok=True)

print(f"Set fitur penuh    : {len(F.ALL_FEATURES)} fitur")
print(f"Set fitur geometri : {len(F.GEOMETRY_FEATURES)} fitur")
print(f"Dibuang (warna)    : {', '.join(F.KINEMATIC_FEATURES)}")

Set fitur penuh    : 19 fitur
Set fitur geometri : 13 fitur
Dibuang (warna)    : vel_mean, vel_std, acc_mean, jerk_mean, fixation_ratio, saccade_ratio


## Muat data dan ekstrak fitur

547 citra, 54 partisipan. Hasil ekstraksi di-cache ke `hasil/fitur.csv` supaya notebook
training dan degradasi memakai matriks yang persis sama.

In [2]:
meta = F.load_metadata()
cache = HASIL / "fitur.csv"

if cache.exists():
    X_df = pd.read_csv(cache)[F.ALL_FEATURES]
    print(f"Fitur dimuat dari cache: {cache.name}")
else:
    X_df = F.feature_frame(meta)
    X_df.to_csv(cache, index=False)
    print(f"Fitur diekstrak dan disimpan ke: {cache.name}")

y = meta.label.values
groups = meta.participant.values

n_asd = meta.groupby("participant").label.max().sum()
n_td = meta.participant.nunique() - n_asd
print(f"\nCitra      : {len(meta)}")
print(f"Partisipan : {meta.participant.nunique()} ({n_asd} ASD, {n_td} TD)")
print(f"Matriks    : {X_df.shape}")

Fitur dimuat dari cache: fitur.csv

Citra      : 547
Partisipan : 54 (26 ASD, 28 TD)
Matriks    : (547, 19)


## Perbandingan lima model pada kedua set fitur

Semua model dijalankan dengan lipatan yang identik, sehingga kolom penuh dan geometri
langsung sebanding baris per baris.

In [3]:
oof = {}
baris = []

for nama_set in ["penuh", "geometri"]:
    X = F.select(X_df, nama_set)
    for nama_model, model in E.build_models().items():
        p = E.oof_probabilities(model, X, y, groups)
        oof[(nama_set, nama_model)] = p
        p_anak, y_anak, _ = E.aggregate_to_participants(p, y, groups)
        baris.append({
            "set_fitur": nama_set,
            "model": nama_model,
            "auc_citra": roc_auc_score(y, p),
            "auc_anak": roc_auc_score(y_anak, p_anak),
        })

tabel_panjang = pd.DataFrame(baris)

tabel = tabel_panjang.pivot(index="model", columns="set_fitur", values="auc_anak")
tabel.columns = [f"anak_{c}" for c in tabel.columns]
tabel["anak_selisih"] = tabel.anak_geometri - tabel.anak_penuh

citra = tabel_panjang.pivot(index="model", columns="set_fitur", values="auc_citra")
tabel["citra_penuh"] = citra.penuh
tabel["citra_geometri"] = citra.geometri

tabel[["citra_penuh", "citra_geometri", "anak_penuh", "anak_geometri", "anak_selisih"]].round(4)

,citra_penuh,citra_geometri,anak_penuh,anak_geometri,anak_selisih
model,,,,,
Gradient Boosting,0.7655,0.7599,0.8626,0.8654,0.0027
Logistic Regression,0.7531,0.7700,0.8448,0.8310,-0.0137
Naive Bayes,0.7753,0.7867,0.8338,0.8585,0.0247
Random Forest,0.7627,0.7597,0.8640,0.8599,-0.0041
SVM (RBF),0.7333,0.7430,0.8489,0.8544,0.0055


## Endpoint utama

Regresi logistik ditetapkan sebagai model pra-tetap sebelum melihat hasil. Dua alasan:
koefisiennya dapat diekspor apa adanya ke produk sehingga inferensi di perangkat hanya
berupa dot product tanpa runtime ML, dan setiap koefisien dapat diperiksa juri satu per satu.

In [4]:
hasil_utama = {}
prob_anak = {}

for nama_set in ["penuh", "geometri"]:
    p = oof[(nama_set, E.PRIMARY_MODEL)]
    p_anak, y_anak, _ = E.aggregate_to_participants(p, y, groups)
    prob_anak[nama_set] = p_anak
    auc = roc_auc_score(y_anak, p_anak)
    ci = E.bootstrap_auc(p_anak, y_anak)
    op = E.operating_point(p_anak, y_anak)
    hasil_utama[nama_set] = {
        "auc_anak": auc,
        "ci95": list(ci),
        "auc_citra": roc_auc_score(y, p),
        "titik_kerja": op,
        "ppv_prevalensi_1persen": E.ppv_at_prevalence(op["sensitivity"], op["specificity"], 0.01),
    }
    print(f"{nama_set:9s}  AUC anak {auc:.4f}  CI 95% [{ci[0]:.4f}, {ci[1]:.4f}]  "
          f"sens {op['sensitivity']:.1%} spes {op['specificity']:.1%}")

delta = E.paired_bootstrap_delta_auc(prob_anak["geometri"], prob_anak["penuh"], y_anak)
print(f"\nSelisih (geometri - penuh): {delta['delta']:+.4f}  "
      f"CI 95% [{delta['ci_low']:+.4f}, {delta['ci_high']:+.4f}]")

penuh      AUC anak 0.8448  CI 95% [0.7296, 0.9439]  sens 92.3% spes 42.9%


geometri   AUC anak 0.8310  CI 95% [0.7117, 0.9327]  sens 100.0% spes 21.4%



Selisih (geometri - penuh): -0.0137  CI 95% [-0.0593, +0.0247]


## Ukuran efek pada 13 fitur geometri

Kalau fitur geometri memang membawa sinyalnya, ukuran efeknya harus tetap besar setelah
enam fitur warna dibuang. Dihitung pada level anak (fitur dirata-ratakan per partisipan
lebih dulu), bukan level citra.

In [5]:
p_level = E.participant_level_frame(X_df, groups, y)
asd_p = p_level[p_level.label == 1]
td_p = p_level[p_level.label == 0]

efek = pd.DataFrame({
    "cohens_d": {f: E.cohens_d(asd_p[f], td_p[f]) for f in F.ALL_FEATURES},
})
efek["kelompok"] = ["geometri" if f in F.GEOMETRY_FEATURES else "warna" for f in efek.index]
efek["abs_d"] = efek.cohens_d.abs()
efek = efek.sort_values("abs_d", ascending=False)

print("Sepuluh efek terbesar:")
print(efek.head(10)[["cohens_d", "kelompok"]].round(3).to_string())
print(f"\nRata-rata |d| geometri : {efek[efek.kelompok == 'geometri'].abs_d.mean():.3f}")
print(f"Rata-rata |d| warna    : {efek[efek.kelompok == 'warna'].abs_d.mean():.3f}")

Sepuluh efek terbesar:
                cohens_d  kelompok
n_active_cells     1.685  geometri
ink_frac           1.500  geometri
span_y             1.478  geometri
grid_entropy       1.471  geometri
radial_mean        1.374  geometri
span_x             1.312  geometri
radial_std         1.294  geometri
vel_mean           1.218     warna
std_y              1.213  geometri
fixation_ratio    -1.150     warna

Rata-rata |d| geometri : 1.068
Rata-rata |d| warna    : 0.800


## Evaluasi aturan keputusan

Kedua kriteria diperiksa persis seperti yang ditetapkan di atas, lalu hasilnya ditulis ke
`hasil/ablasi.json` supaya setiap angka yang dikutip di paper bisa dilacak ke file.

In [6]:
AMBANG_AUC = 0.80
MARGIN_NONINFERIORITAS = -0.05

auc_geometri = hasil_utama["geometri"]["auc_anak"]
k1 = auc_geometri >= AMBANG_AUC
k2 = delta["ci_low"] > MARGIN_NONINFERIORITAS
lulus = k1 and k2

print(f"K1  AUC anak geometri {auc_geometri:.4f} >= {AMBANG_AUC}          : {'LULUS' if k1 else 'GAGAL'}")
print(f"K2  batas bawah CI selisih {delta['ci_low']:+.4f} > {MARGIN_NONINFERIORITAS}    : {'LULUS' if k2 else 'GAGAL'}")
print()
print(f"KEPUTUSAN: {'Jalan B — kunci model geometri 13 fitur' if lulus else 'Jalan A — verifikasi encoding Carette dulu'}")

payload = {
    "pertanyaan": "Apakah membuang 6 fitur turunan warna merugikan performa secara berarti?",
    "aturan_keputusan": {
        "K1_ambang_auc_anak": AMBANG_AUC,
        "K2_margin_noninferioritas": MARGIN_NONINFERIORITAS,
        "ditetapkan": "sebelum melihat hasil",
    },
    "n_citra": int(len(meta)),
    "n_partisipan": int(meta.participant.nunique()),
    "n_fitur_penuh": len(F.ALL_FEATURES),
    "n_fitur_geometri": len(F.GEOMETRY_FEATURES),
    "fitur_dibuang": F.KINEMATIC_FEATURES,
    "fitur_geometri": F.GEOMETRY_FEATURES,
    "model_pra_tetap": E.PRIMARY_MODEL,
    "hasil": {k: {
        "auc_anak": round(v["auc_anak"], 4),
        "ci95": [round(c, 4) for c in v["ci95"]],
        "auc_citra": round(v["auc_citra"], 4),
        "titik_kerja": {kk: round(vv, 4) for kk, vv in v["titik_kerja"].items()},
        "ppv_prevalensi_1persen": round(v["ppv_prevalensi_1persen"], 4),
    } for k, v in hasil_utama.items()},
    "selisih_berpasangan": {k: round(v, 4) for k, v in delta.items()},
    "auc_semua_model": tabel_panjang.round(4).to_dict(orient="records"),
    "cohens_d": {f: round(float(efek.loc[f, "cohens_d"]), 3) for f in efek.index},
    "K1_lulus": bool(k1),
    "K2_lulus": bool(k2),
    "keputusan": "geometri_13_fitur" if lulus else "perlu_verifikasi_encoding",
}

out = HASIL / "ablasi.json"
out.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n")
print(f"\nDitulis ke {out}")

K1  AUC anak geometri 0.8310 >= 0.8          : LULUS
K2  batas bawah CI selisih -0.0593 > -0.05    : GAGAL

KEPUTUSAN: Jalan A — verifikasi encoding Carette dulu

Ditulis ke /Users/timotiuspatrick/Documents/Patrick/Kuliah & Coding/Dataton/project/research/hasil/ablasi.json
